# 01 – CART from scratch (course annexes)
We re-implement CART with the **Gini index** as in Chapter 4, reproduce the hand calculations of the two annexes (Golf, Student success), and validate against scikit-learn.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path('..') / 'src'))
import pandas as pd
from sklearn.tree import DecisionTreeClassifier, export_text
from ensemble_disease.cart import CARTClassifier, gini_table
from ensemble_disease.toy_data import GOLF, STUDENT

## 1. Golf dataset – root split
Gini(S) = 1 − Σ pᵢ²; the split with the lowest **weighted Gini** wins.

In [2]:
t = gini_table(GOLF, ['Outlook','Temp','Humidity','Wind'], 'Decision')
pd.Series(t).round(3).sort_values()

Outlook     0.343
Humidity    0.367
Wind        0.429
Temp        0.440
dtype: float64

Annex values: Outlook 0.342, Humidity 0.367, Wind 0.428, Temp 0.439. Our values differ only in the 3rd decimal (the slides round intermediate terms). **Outlook** is the root in both.

In [3]:
golf_tree = CARTClassifier().fit(GOLF, 'Decision')
print(golf_tree.rules())
print('train accuracy:', golf_tree.score(GOLF))

IF Outlook = Overcast THEN Decision = Yes  {'Yes': 4}
IF Outlook = Rain AND Wind = Strong THEN Decision = No  {'No': 2}
IF Outlook = Rain AND Wind = Weak THEN Decision = Yes  {'Yes': 3}
IF Outlook = Sunny AND Humidity = High THEN Decision = No  {'No': 3}
IF Outlook = Sunny AND Humidity = Normal THEN Decision = Yes  {'Yes': 2}
train accuracy: 1.0


## 2. Correction of a slide typo (Sunny branch, Wind)
The annex shows Gini(Sunny, Wind=Weak)=0.266 and Gini(Sunny, Wind=Strong)=0.2. The correct values are:

In [4]:
from ensemble_disease.cart import gini
sunny = GOLF[GOLF.Outlook=='Sunny']
for w, g in sunny.groupby('Wind'):
    print(w, round(gini(g['Decision']), 3))
print('weighted:', round(3/5*0.444+2/5*0.5, 3), '(0.466 in the slide, so the final value is right)')

Strong 0.5
Weak 0.444
weighted: 0.466 (0.466 in the slide, so the final value is right)


## 3. Student success dataset
Root: Attendance (0.3265), then StudyTime (High) / Homework (Low), as in the course.

In [5]:
stu_cols = list(STUDENT.columns[:-1])
print(pd.Series(gini_table(STUDENT, stu_cols, 'Success')).round(4).sort_values())
stu_tree = CARTClassifier().fit(STUDENT, 'Success')
print(stu_tree.rules())
print('train accuracy:', stu_tree.score(STUDENT))

Attendance       0.3265
StudyTime        0.4490
PreviousGrade    0.4490
Homework         0.4490
dtype: float64
IF Attendance = High AND StudyTime = High THEN Success = YES  {'YES': 4}
IF Attendance = High AND StudyTime = Low AND PreviousGrade = High THEN Success = YES  {'YES': 1}
IF Attendance = High AND StudyTime = Low AND PreviousGrade = Low AND Homework = High THEN Success = NO  {'NO': 1}
IF Attendance = High AND StudyTime = Low AND PreviousGrade = Low AND Homework = Low THEN Success = YES  {'YES': 1}
IF Attendance = Low AND Homework = High AND PreviousGrade = High THEN Success = YES  {'YES': 2}
IF Attendance = Low AND Homework = High AND PreviousGrade = Low THEN Success = NO  {'NO': 1}
IF Attendance = Low AND Homework = Low THEN Success = NO  {'NO': 4}
train accuracy: 1.0


### Gini table at every split (useful for the report)

In [6]:
for path, table in stu_tree.split_log:
    best = min(table, key=table.get)
    print(f'{path:55s} -> {best:14s}', {k: round(v,4) for k,v in table.items()})

root                                                    -> Attendance     {'Attendance': 0.3265, 'StudyTime': 0.449, 'PreviousGrade': 0.449, 'Homework': 0.449}
root/Attendance=High                                    -> StudyTime      {'StudyTime': 0.1905, 'PreviousGrade': 0.2143, 'Homework': 0.2143}
root/Attendance=High/StudyTime=Low                      -> PreviousGrade  {'PreviousGrade': 0.3333, 'Homework': 0.3333}
root/Attendance=High/StudyTime=Low/PreviousGrade=Low    -> Homework       {'Homework': 0.0}
root/Attendance=Low                                     -> Homework       {'StudyTime': 0.4048, 'PreviousGrade': 0.2857, 'Homework': 0.1905}
root/Attendance=Low/Homework=High                       -> PreviousGrade  {'StudyTime': 0.3333, 'PreviousGrade': 0.0}


## 4. Validation against scikit-learn
scikit-learn builds **binary** trees on one-hot features, so structures differ (multiway vs binary), but both reach the same purity and use the same root.

In [7]:
X = pd.get_dummies(GOLF.drop(columns='Decision'))
sk = DecisionTreeClassifier(criterion='gini', random_state=42).fit(X, GOLF.Decision)
print(export_text(sk, feature_names=list(X.columns)))
print('sklearn train acc:', sk.score(X, GOLF.Decision), '| ours:', golf_tree.score(GOLF))

|--- Outlook_Overcast <= 0.50
|   |--- Humidity_Normal <= 0.50
|   |   |--- Outlook_Rain <= 0.50
|   |   |   |--- class: No
|   |   |--- Outlook_Rain >  0.50
|   |   |   |--- Wind_Strong <= 0.50
|   |   |   |   |--- class: Yes
|   |   |   |--- Wind_Strong >  0.50
|   |   |   |   |--- class: No
|   |--- Humidity_Normal >  0.50
|   |   |--- Wind_Strong <= 0.50
|   |   |   |--- class: Yes
|   |   |--- Wind_Strong >  0.50
|   |   |   |--- Outlook_Sunny <= 0.50
|   |   |   |   |--- class: No
|   |   |   |--- Outlook_Sunny >  0.50
|   |   |   |   |--- class: Yes
|--- Outlook_Overcast >  0.50
|   |--- class: Yes

sklearn train acc: 1.0 | ours: 1.0


## Take-aways
- Reproduces every hand calculation of the annexes (up to rounding).
- A fully grown tree reaches 100% train accuracy on 14 rows: this is **overfitting**, which motivates bagging / boosting / stacking (next steps).
- Course trees are multiway on nominal features; CART proper (scikit-learn) is binary.